In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from imblearn.over_sampling import RandomOverSampler
from sklearn.preprocessing import LabelEncoder
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 读取数据
train_df = pd.read_csv('new_cicddos2019_dataset.csv')


# 移除无关列
train_df.drop(columns=['Unnamed: 0', 'Class'], inplace=True)


# 检查是否有缺失值
print("Train missing values:", train_df.isnull().sum().sum())


# One-hot 编码列
# cols = ['proto', 'state', 'service']
# 
# def one_hot(df, cols):
#     for col in cols:
#         dummies = pd.get_dummies(df[col], prefix=col, drop_first=False)
#         df = pd.concat([df, dummies], axis=1)
#         df.drop(col, axis=1, inplace=True)
#     return df

# 合并数据进行统一处理
combined_data = train_df

# 分离目标变量
target = combined_data.pop('Label')

# One-hot 编码
# combined_data = one_hot(combined_data, cols)

# Min-Max 归一化
def normalize(df):
    result = df.copy()
    for col in df.columns:
        max_val = df[col].max()
        min_val = df[col].min()
        if max_val > min_val:
            result[col] = (df[col] - min_val) / (max_val - min_val)
    return result

# 归一化数据
normalized_data = normalize(combined_data)

# 添加目标列
normalized_data['Class'] = target

# 检查是否有缺失值
assert not normalized_data.isnull().values.any(), "归一化后的数据存在缺失值"

# 分离特征和标签
X = normalized_data.drop(columns=['Class']).values
y = normalized_data['Class'].values
print(X.shape)
# 将目标变量转换为整数编码
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

Train missing values: 0
(203880, 77)


In [10]:
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
# 设置超参数
k = 10  # 交叉验证的折数
models = {
    "KNN": KNeighborsClassifier(n_neighbors=1),
    "Decision Tree": DecisionTreeClassifier(random_state=40)
}

# 初始化交叉验证
kfold = StratifiedKFold(n_splits=k, shuffle=True, random_state=40)

# 交叉验证
for model_name, model in models.items():
    print(f"\n===== 训练 {model_name} =====")
    oos_accuracies = []  # 记录每折的准确率
    last_fold_y_true, last_fold_y_pred = [], []  # 最后一折的预测结果

    for fold, (train_idx, test_idx) in enumerate(kfold.split(X, y_encoded), start=1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

        # 训练模型
        model.fit(X_train, y_train)

        # 预测
        y_pred = model.predict(X_test)

        # 计算准确率
        acc = accuracy_score(y_test, y_pred)
        oos_accuracies.append(acc)
        print(f"  Fold {fold} Accuracy: {acc:.4f}")

        # 记录最后一折的结果
        if fold == k:
            last_fold_y_true, last_fold_y_pred = y_test, y_pred

    # 输出最终平均准确率
    mean_acc = np.mean(oos_accuracies)
    print(f"\n{model_name} 平均准确率: {mean_acc:.4f}")

    # 分类类别
    categories = ['1', '2', '3', '4', '5', '6','7', '8', '9', '10','11', '12', '13', '14', '15', '16','17', '18']

    # 混淆矩阵（最后一折的结果）
    last_cm = confusion_matrix(last_fold_y_true, last_fold_y_pred, labels=range(18))

    print("\nLast Fold Confusion Matrix:")
    print(last_cm)

    report = classification_report(last_fold_y_true, last_fold_y_pred, target_names=categories)
    print("\nClassification Report:")
    print(report)

    # 从混淆矩阵提取所有类别的统计信息
    total_samples = last_cm.sum()  # 总样本数
    correct_predictions = np.trace(last_cm)  # 对角线元素的和，即正确分类的总数

    # 总体准确率（直接计算）
    overall_accuracy = correct_predictions / total_samples

    # 初始化总体指标
    overall_TP = 0
    overall_FN = 0
    overall_FP = 0
    overall_TN = 0


    # 重新计算分类报告中的 TP、FP、FN、TN
    detection_rates = {}
    false_positive_rates = {}

    for i, category in enumerate(categories):
        TP = last_cm[i, i]
        FN = last_cm[i, :].sum() - TP
        FP = last_cm[:, i].sum() - TP
        TN = total_samples - (TP + FP + FN)

        if category != "1":  # 统计攻击类型的总 TP 和 FN
            overall_TP += TP
            overall_FN += FN
        else:  # 统计正常类型的 TN 和 FP
            overall_TN += TN
            overall_FP += FP

        # 每类检测率和误报率
        detection_rates[category] = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        false_positive_rates[category] = FP / (FP + TN) if (FP + TN) > 0 else 0.0

        print(f"Category: {category}")
        print(f"  Detection Rate (DR): {detection_rates[category]:.4f}")
        print(f"  False Positive Rate (FPR): {false_positive_rates[category]:.4f}")

    # 总体检测率（攻击类型的 DR）和误报率（Normal 的 FPR）
    overall_dr = overall_TP / (overall_TP + overall_FN) if (overall_TP + overall_FN) > 0 else 0.0
    overall_fpr = overall_FP / (overall_FP + overall_TN) if (overall_FP + overall_TN) > 0 else 0.0

    print("\nOverall Metrics:")
    print(f"  Accuracy (Acc): {overall_accuracy:.4f}")
    print(f"  Detection Rate (DR): {overall_dr:.4f}")
    print(f"  False Positive Rate (FPR): {overall_fpr:.4f}")


===== 训练 KNN =====
  Fold 1 Accuracy: 0.8429
  Fold 2 Accuracy: 0.8433
  Fold 3 Accuracy: 0.8411
  Fold 4 Accuracy: 0.8442
  Fold 5 Accuracy: 0.8429
  Fold 6 Accuracy: 0.8439
  Fold 7 Accuracy: 0.8433
  Fold 8 Accuracy: 0.8441
  Fold 9 Accuracy: 0.8445
  Fold 10 Accuracy: 0.8436

KNN 平均准确率: 0.8434

Last Fold Confusion Matrix:
[[3716    5    0    0    5    0    0    0    0    0    0    0    0    0
     0    0    1    0]
 [   1  154   11    4    3    0   11    1  118   47    0    5    0    1
    10    0    1    0]
 [   0   30   10    2    0    0    6    0   90    4    0    0    1    1
     0    0    0    0]
 [   0   11    1  100    0    3    4    2    9  448    0    4    2    1
    37    0    0    0]
 [   6    5    0    0 4605    0    0    0    0    1    0    0    1    1
     0    2    1    1]
 [   2    0    0    0    0    4    3    0    0    7    5   34    0    0
     4    0    0    0]
 [   0   24    8    3    0    1  140    0   64   20    5    6    0    1
     0    0    0    0]
 [   0